# Experiment 2 — learning curves

Metric vs **training-set size** (nested subsets). One line per method, averaged over all included datasets; linear x axis. The **relative** variants divide each method's curve by its own best value — 1.0 means "at this size the method already reaches its top performance".

Figures → `figures/experiment2/` (wiped on rerun).

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, learning_curve, imbalance_curve,
    metric_boxplots, metric_bars, median_time_bars, rank_heatmap, rank_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment2')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
pd_df  = load_summary(SUMMARY_DIR, experiment='experiment2', task='pd')
try:
    lgd_df = load_summary(SUMMARY_DIR, experiment='experiment2', task='lgd')
except FileNotFoundError:
    lgd_df = None
print(f'PD: {pd_df["method"].nunique()} methods, {pd_df["dataset"].nunique()} datasets, '
      f'{pd_df["sweep_value"].nunique()} sweep points')

## PD — absolute

In [ ]:
learning_curve(pd_df, 'AUC',   task_name='PD', out_dir=FIGURES_DIR / 'pd')
learning_curve(pd_df, 'Brier', task_name='PD', out_dir=FIGURES_DIR / 'pd')   # lower = better

## PD — relative to each method's own top performance

In [ ]:
learning_curve(pd_df, 'AUC', task_name='PD', relative=True, out_dir=FIGURES_DIR / 'pd')

## LGD — absolute

In [ ]:
if lgd_df is not None:
    learning_curve(lgd_df, 'R2',   task_name='LGD', out_dir=FIGURES_DIR / 'lgd')
    learning_curve(lgd_df, 'RMSE', task_name='LGD', out_dir=FIGURES_DIR / 'lgd')

## LGD — relative to each method's own top performance

In [ ]:
if lgd_df is not None:
    learning_curve(lgd_df, 'R2', task_name='LGD', relative=True, out_dir=FIGURES_DIR / 'lgd')

## Data efficiency — rows needed to reach 95% of each method's own plateau

In [ ]:
import pandas as _pd
g = (pd_df.groupby(['method','sweep_value'])['metric.AUC_mean'].mean().reset_index())
rows = {}
for m, gg in g.groupby('method'):
    gg = gg.sort_values('sweep_value')
    target = gg['metric.AUC_mean'].max() * 0.95
    hit = gg[gg['metric.AUC_mean'] >= target]
    rows[m] = int(hit['sweep_value'].iloc[0]) if len(hit) else None
display(_pd.Series(rows, name='rows to reach 95% of own max AUC').sort_values())